# Get comments, score predictions and parse them into a json

## Codebase

In [1]:
import requests
from dotenv import load_dotenv
import os
import re
from itertools import combinations
import json
from pathlib import Path


### Youtube

Gets all comments under a youtube

In [2]:
# Load the API key from the .env file
load_dotenv()
API_KEY = os.getenv("API_KEY")


In [3]:
def get_video_id_from_url(url):
    """
    Extracts the video ID from a YouTube URL.
    """
    if "v=" in url:
        return url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]
    else:
        raise ValueError("Invalid YouTube URL")

In [4]:
def get_all_comments_once(api_key: str, video_id: str) -> list[dict]:
    url = "https://www.googleapis.com/youtube/v3/commentThreads"
    comments = []
    page_token = None

    # check if the video ID is valid or if it is a URL and extract the video ID
    if "youtube.com" in video_id or "youtu.be" in video_id:
        video_id = get_video_id_from_url(video_id)

    while True:
        params = {
            "key": api_key,
            "part": "snippet",
            "videoId": video_id,
            "maxResults": 100,
            "order": "time",
            "textFormat": "plainText",
        }

        if page_token:
            params["pageToken"] = page_token

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()

        for thread in data.get("items", []):
            comment = thread["snippet"]["topLevelComment"]
            snippet = comment["snippet"]

            comments.append({
                "id": comment["id"],
                "author": snippet.get("authorDisplayName"),
                "text": snippet.get("textDisplay"),
                "likes": snippet.get("likeCount", 0),
                "published_at": snippet.get("publishedAt"),
            })

        page_token = data.get("nextPageToken")

        if not page_token:
            break

    return comments

### Data processing

In [5]:
def find_team_mentions_in_text(text, teams):
    results = []

    for team, aliases in teams.items():
        for alias in aliases:
            alias_text = alias.replace("_", " ")

            pattern = (
                r"(?<![A-Za-z0-9])"
                + re.escape(alias_text)
                + r"(?![A-Za-z0-9])"
            )

            for match in re.finditer(pattern, text, flags=re.IGNORECASE):
                results.append({
                    "team": team,
                    "alias": alias,
                    "matched_text": match.group(),
                    "start": match.start(),
                    "end": match.end(),
                })

    return sorted(results, key=lambda x: x["start"])

In [6]:
def parse_weekly_games(weekly_games):
    fixtures = {}

    for game in weekly_games:
        home, away = [x.strip() for x in game.split(" - ")]
        fixtures[frozenset([home, away])] = {
            "fixture": game,
            "home_team": home,
            "away_team": away,
        }

    return fixtures

In [7]:
def find_numbers_in_line(line):
    matches = []

    for match in re.finditer(r"\b\d{1,2}\b", line):
        matches.append({
            "value": int(match.group()),
            "start": match.start(),
            "end": match.end(),
        })

    return matches

In [8]:
def find_fixture_scores(comment, teams, weekly_games):
    text = comment["text"] if isinstance(comment, dict) else comment
    fixtures = parse_weekly_games(weekly_games)

    results = []

    for line_number, line in enumerate(text.splitlines(), start=1):
        mentions = find_team_mentions_in_text(line, teams)

        # Deduplicate teams in same line
        seen = set()
        team_mentions = []

        for mention in mentions:
            if mention["team"] not in seen:
                seen.add(mention["team"])
                team_mentions.append(mention)

        if len(team_mentions) != 2:
            continue

        first, second = team_mentions
        pair = frozenset([first["team"], second["team"]])

        if pair not in fixtures:
            continue

        numbers = find_numbers_in_line(line)

        if len(numbers) < 2:
            continue

        fixture = fixtures[pair]

        # Assign scores based on nearest number after each team mention
        first_score = next(
            (n for n in numbers if n["start"] >= first["end"]),
            None
        )
        second_score = next(
            (n for n in numbers if n["start"] >= second["end"]),
            None
        )

        if first_score is None or second_score is None:
            continue

        scores_by_team = {
            first["team"]: first_score["value"],
            second["team"]: second_score["value"],
        }

        # find the winner based on the scores
        if scores_by_team[first["team"]] > scores_by_team[second["team"]]:
            winner = "home"
        elif scores_by_team[first["team"]] < scores_by_team[second["team"]]:
            winner = "away"
        else:
            winner = "draw"

        results.append({
            "fixture": fixture["fixture"],
            "home_team": fixture["home_team"],
            "away_team": fixture["away_team"],
            "home_score": scores_by_team[fixture["home_team"]],
            "away_score": scores_by_team[fixture["away_team"]],
            "winner": winner,
            "line_number": line_number,
            "matched_text": line.strip(),
        })

    return results

In [9]:
def format_comment_predictions(comment, raw_scores, weekly_games):
    expected_fixtures = set(weekly_games)
    fixtures = parse_weekly_games(weekly_games)

    found_by_fixture = {
        score["fixture"]: score
        for score in raw_scores
    }

    predictions = {}
    missing_games = []

    for fixture in weekly_games:
        result = found_by_fixture.get(fixture)
        fixture_info = fixtures[frozenset(fixture.split(" - "))]

        if result is None:
            missing_games.append(fixture)
            predictions[fixture] = {
                "fixture": fixture,
                "found": False,
                "prediction": None,
                "home_team": fixture_info["home_team"],
                "away_team": fixture_info["away_team"],
                "home_score": None,
                "away_score": None,
                "winner": None,
            }
            continue

        predictions[fixture] = {
            "fixture": fixture,
            "found": True,
            "prediction": f"{result['home_score']}-{result['away_score']}",
            "home_team": result["home_team"],
            "away_team": result["away_team"],
            "home_score": result["home_score"],
            "away_score": result["away_score"],
            "winner": result["winner"],
        }

    return {
        "comment_id": comment["id"],
        "author": comment["author"],
        "published_at": comment.get("published_at"),
        "total_games_expected": len(expected_fixtures),
        "total_games_found": len(expected_fixtures) - len(missing_games),
        "all_games_found": len(missing_games) == 0,
        "missing_games": missing_games,
        "predictions": predictions,
    }


In [10]:
#raw_scores = find_fixture_scores(comments[1], dict_of_teams, weekly_games)
#readable_comment = format_comment_predictions(comments[1], raw_scores, weekly_games)

#print(json.dumps(readable_comment, indent=2, ensure_ascii=False))

In [11]:
def save_week_predictions(
    comments,
    dict_of_teams,
    weekly_games,
    week_id,
    output_dir="data/predictions",
    min_games_found=3,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_comments = []

    for comment in comments:
        raw_scores = find_fixture_scores(comment, dict_of_teams, weekly_games)
        readable_comment = format_comment_predictions(
            comment,
            raw_scores,
            weekly_games
        )

        if readable_comment["total_games_found"] >= min_games_found:
            saved_comments.append(readable_comment)

    week_data = {
        "week_id": week_id,
        "fixtures": weekly_games,
        "total_comments_saved": len(saved_comments),
        "comments": saved_comments,
    }

    output_path = output_dir / f"{week_id}.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(week_data, f, indent=2, ensure_ascii=False)

    return output_path


In [12]:
# function wrapper to make it easier to call from the notebook
def func_wrapper(week_id,API_KEY, VIDEO_ID, dict_of_teams, weekly_games):
    comments = get_all_comments_once(API_KEY, VIDEO_ID)

    print(f"Downloaded {len(comments)} comments")

    output_path = save_week_predictions(
        comments=comments,
        dict_of_teams=dict_of_teams,
        weekly_games=weekly_games,
        week_id=week_id,
    )
    return output_path


# Run the code

## User defined values

Only for prototype, has to automate it somehow for the future.

### League data

In [13]:
dict_of_teams = {
    "Arsenal": ["Arsenal"],
    "Aston_Villa": ["Aston_Villa","Villa"],
    "Bournemouth": ["Bournemouth"],
    "Brentford": ["Brentford"],
    "Brighton": ["Brighton"],
    "Burnley": ["Burnley"],
    "Chelsea": ["Chelsea"],
    "Crystal_Palace": ["Crystal_Palace", "Palace"],
    "Everton": ["Everton"],
    "Fulham": ["Fulham"],
    "Leeds": ["Leeds_United", "Leeds"],
    "Liverpool": ["Liverpool", "Pool"],
    "Manchester_City": ["Manchester City", "Man City"],
    "Manchester_United": ["Manchester United", "MU", "Man United"],
    "Newcastle": ["Newcastle United", "Newcastle", "Newcastle Utd"],
    "Nottingham_Forest": ["Nottingham Forest","Nottingham", "Forest"],
    "Sunderland": ["Sunderland"],
    "Tottenham": ["Tottenham Hotspurs","Tottenham", "Spurs"],
    "West_Ham": ["West Ham"],
    "Wolves": ["Wolverhampton Wanderers", "Wolves", "Wolverhampton"],
}

### Week related

In [14]:
weekly_games = [
    "Leeds - Nottingham_Forest",
    "Manchester_United - Tottenham",
    "Bournemouth - Aston_Villa",
    "Arsenal - Sunderland",
    "Burnley - West_Ham",
    "Fulham - Everton",
    "Wolves - Chelsea",
    "Newcastle - Brentford",
    "Brighton - Crystal_Palace",
    "Liverpool - Manchester_City"
]

### Video related

In [15]:
VIDEO_URL = "https://www.youtube.com/watch?v=wWsEOs-p5os"
VIDEO_ID = get_video_id_from_url(VIDEO_URL)

WEEK_ID = "2026_week_06"



### Run these to get json

In [16]:
output_path = func_wrapper(
    week_id=WEEK_ID,
    API_KEY=API_KEY,
    VIDEO_ID=VIDEO_ID,
    dict_of_teams=dict_of_teams,
    weekly_games=weekly_games
)
print(f"Saved predictions to {output_path}")

Downloaded 206 comments
Saved predictions to data/predictions/2026_week_06.json


# Point system

- 5 points for the correct outcome
- 10 points for bang on scores